# P3-07: MKI67 Transcript–Protein Validation & IDO1 Spatial Analysis

**Goal:** (1) Validate Ki-67 protein (IMC) against MKI67 transcript (Xenium) in
co-registered GBM tissue to confirm cross-platform concordance. (2) Assess IDO1
spatial expression relative to Ki-67+ proliferating cells and nicotinic acid MSI
signal, testing the hypothesis that tryptophan catabolism (IDO1) drives
compensatory NAD+ synthesis via the Preiss-Handler pathway (nicotinic acid).

**Approach:**
- Match Xenium cells → IMC cells via cKDTree in shared H&E coordinate space
- Match Xenium cells → MSI pixels via cKDTree in shared H&E coordinate space
- Compute MKI67 transcript vs Ki-67 protein Spearman correlation per ROI
- Assess IDO1 expression in Ki-67+ vs Ki-67- neighborhoods at Edge vs Core
- Correlate IDO1 expression with nicotinic acid MSI intensity

**Inputs:**
- RCTD-annotated Xenium AnnData (`PATHS.rctd_adata_dir`)
- IMC AnnData with Ki-67 protein (`PATHS.imc_adata_dir`)
- Pre-warped MSI AnnData (`PATHS.msi_aligned_mets_dir`)

**Outputs:**
- `outputs/07_mki67_ido1/07_mki67_protein_transcript_corr.csv`
- `outputs/07_mki67_ido1/07_mki67_scatter_per_roi.pdf`
- `outputs/07_mki67_ido1/07_ido1_ki67_association.csv`
- `outputs/07_mki67_ido1/07_ido1_nicotinic_corr.csv`
- `outputs/07_mki67_ido1/07_summary_figure.pdf`

In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, '.')
from P3_config import *

import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial import cKDTree
from scipy import stats
from statsmodels.stats.multitest import multipletests
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

log_step('Notebook P3-07: MKI67 Transcript-Protein Validation & IDO1 Analysis')
outdir = ensure_output_dir('07_mki67_ido1')

## 1. Configuration

In [ ]:
# --- Matching parameters (same as P3-06) ---
DISTANCE_CUTOFF_IMC = 15.0    # H&E pixels (~30 µm) for Xenium-to-IMC matching
DISTANCE_CUTOFF_MSI = 15.0    # H&E pixels for Xenium-to-MSI matching
MIN_CELLS_PER_GROUP = 10      # Min cells for statistical tests

# --- Genes of interest ---
XENIUM_GENES = ['MKI67', 'IDO1']

# --- IMC markers ---
KI67_MARKER = 'KI67'          # from P3_config
IMC_KI67_COFACTOR = 5         # arcsinh cofactor

# --- Nicotinic acid channels in MSI ---
# From ki67_msi_correlation_results: Nicotinic acid_1 showed edge enrichment
# We'll search for 'nicotinic' or 'Nicotinic' in var_names
NICOTINIC_SEARCH_TERMS = ['nicotinic', 'Nicotinic', '122.024']  # niacin [M-H]- = 122.0248

print(f'Target genes: {XENIUM_GENES}')
print(f'GBM ROIs: {len(GBM_ROIS)}')

## 2. Load and Match Xenium Cells to IMC Cells

Both Xenium (via RCTD AnnData) and IMC data have coordinates in H&E space.
We use cKDTree to find the nearest IMC cell for each Xenium cell.

**Coordinate spaces:**
- Xenium RCTD AnnData: `adata.obsm['spatial_he']` = VALIS-transformed H&E coords
  (added by `add_spatial_he_to_rctd_adatas.py`: Xenium µm → DAPI px → H&E px)
- IMC AnnData: `adata.obsm['spatial_warped_full_he']` = QuPath-registered H&E coords

Both must be in H&E pixel space for matching to work correctly.

In [ ]:
log_step('Loading Xenium and IMC data for GBM ROIs')

from pathlib import Path

IMC_DIR = Path(PATHS.imc_adata_dir)
RCTD_DIR = Path(PATHS.rctd_adata_dir)
MSI_DIR = Path(PATHS.msi_aligned_mets_dir)

# ── 2a. Probe first files to discover coordinate keys and gene availability ──
first_roi = list(GBM_ROIS.keys())[0]
first_sample = GBM_ROIS[first_roi]

# IMC
imc_test = ad.read_h5ad(IMC_DIR / f'adata_{first_roi}.h5ad', backed='r')
print(f'IMC example: {imc_test.shape}')
print(f'  .obsm keys: {list(imc_test.obsm.keys())}')
print(f'  .var_names sample: {list(imc_test.var_names[:5])} ...')
print(f'  KI67 in var_names: {KI67_MARKER in imc_test.var_names}')
if KI67_MARKER not in imc_test.var_names:
    print(f'  Looking for Ki67... {[v for v in imc_test.var_names if "ki" in v.lower() or "67" in v]}')

# Xenium/RCTD
rctd_test = ad.read_h5ad(RCTD_DIR / f'adata_{first_sample}_with_rctd.h5ad', backed='r')
print(f'\nXenium/RCTD example: {rctd_test.shape}')
print(f'  .obsm keys: {list(rctd_test.obsm.keys())}')
for gene in XENIUM_GENES:
    found = gene in rctd_test.var_names
    print(f'  {gene} in var_names: {found}')
    if not found:
        # Case-insensitive search
        matches = [v for v in rctd_test.var_names if gene.lower() in v.lower()]
        print(f'    Fuzzy matches: {matches}')

# Determine coordinate keys — both MUST be in H&E pixel space
# Xenium: prefer spatial_he (VALIS-warped), added by add_spatial_he_to_rctd_adatas.py
# IMC: prefer spatial_warped_full_he (QuPath-registered)
XENIUM_HE_CANDIDATES = ['spatial_he', 'spatial_warped_full_he']
IMC_HE_CANDIDATES = ['spatial_warped_full_he', 'spatial_he']

XENIUM_COORD_KEY = None
for key in XENIUM_HE_CANDIDATES:
    if key in rctd_test.obsm:
        XENIUM_COORD_KEY = key
        break
if XENIUM_COORD_KEY is None:
    raise KeyError(
        f'No H&E-space coordinates found in Xenium RCTD AnnData. '
        f'Available obsm keys: {list(rctd_test.obsm.keys())}. '
        f'Run add_spatial_he_to_rctd_adatas.py first to add spatial_he.'
    )

IMC_COORD_KEY = None
for key in IMC_HE_CANDIDATES:
    if key in imc_test.obsm:
        IMC_COORD_KEY = key
        break
if IMC_COORD_KEY is None:
    raise KeyError(
        f'No H&E-space coordinates found in IMC AnnData. '
        f'Available obsm keys: {list(imc_test.obsm.keys())}.'
    )

print(f'\nUsing Xenium coords: obsm["{XENIUM_COORD_KEY}"]')
print(f'Using IMC coords:    obsm["{IMC_COORD_KEY}"]')

# Quick sanity check on coordinate ranges
xen_sample = rctd_test.obsm[XENIUM_COORD_KEY][:100]
imc_sample = imc_test.obsm[IMC_COORD_KEY][:100]
print(f'\n  Xenium sample range: X [{xen_sample[:,0].min():.0f}, {xen_sample[:,0].max():.0f}]  '
      f'Y [{xen_sample[:,1].min():.0f}, {xen_sample[:,1].max():.0f}]')
print(f'  IMC sample range:    X [{imc_sample[:,0].min():.0f}, {imc_sample[:,0].max():.0f}]  '
      f'Y [{imc_sample[:,1].min():.0f}, {imc_sample[:,1].max():.0f}]')

In [ ]:
# ── 2b. Match Xenium cells to nearest IMC cell per ROI ──
log_step('Matching Xenium → IMC cells per ROI')

matched_records = []
n_skipped = 0

for roi_name, sample_name in GBM_ROIS.items():
    # Determine region from sample name
    region = 'Edge' if 'Edge' in sample_name else 'Core'
    
    # Load IMC
    imc_path = IMC_DIR / f'adata_{roi_name}.h5ad'
    if not imc_path.exists():
        print(f'  ✗ IMC not found: {roi_name}')
        n_skipped += 1
        continue
    imc = ad.read_h5ad(imc_path)
    
    # Load Xenium/RCTD
    rctd_path = RCTD_DIR / f'adata_{sample_name}_with_rctd.h5ad'
    if not rctd_path.exists():
        print(f'  ✗ RCTD not found: {sample_name}')
        n_skipped += 1
        continue
    rctd = ad.read_h5ad(rctd_path)
    
    # Get coordinates
    imc_coords = imc.obsm[IMC_COORD_KEY]
    xen_coords = rctd.obsm[XENIUM_COORD_KEY]
    
    # ── Sanity check: are coordinate ranges compatible? ──
    imc_range = np.ptp(imc_coords, axis=0)
    xen_range = np.ptp(xen_coords, axis=0)
    # If ranges differ by >10x, coordinates are likely in different spaces
    ratio = np.max(xen_range) / (np.max(imc_range) + 1e-9)
    if ratio > 10 or ratio < 0.1:
        print(f'  ⚠ {roi_name}: Coordinate scale mismatch! Xenium range={xen_range}, IMC range={imc_range}')
        print(f'    Ratio = {ratio:.1f}x — these may be in different coordinate spaces')
        print(f'    → Check if Xenium needs VALIS transform to H&E space')
        n_skipped += 1
        continue
    
    # Build KDTree on IMC, query from Xenium
    imc_tree = cKDTree(imc_coords)
    distances, imc_indices = imc_tree.query(xen_coords, k=1)
    
    # Filter by distance cutoff
    valid_mask = distances <= DISTANCE_CUTOFF_IMC
    n_valid = valid_mask.sum()
    
    if n_valid < MIN_CELLS_PER_GROUP:
        print(f'  ✗ {roi_name}: Only {n_valid} valid matches (< {MIN_CELLS_PER_GROUP})')
        n_skipped += 1
        continue
    
    # Extract gene expression for matched Xenium cells
    xen_valid_idx = np.where(valid_mask)[0]
    imc_valid_idx = imc_indices[valid_mask]
    
    # Ki-67 protein from IMC
    ki67_col = KI67_MARKER
    if ki67_col not in imc.var_names:
        # Try fallback: use index
        ki67_col = imc.var_names[KI67_INDEX]
    
    if 'arcsinh' in imc.layers:
        ki67_protein = np.asarray(imc[imc_valid_idx, ki67_col].layers['arcsinh']).flatten()
    else:
        ki67_protein = np.asarray(imc[imc_valid_idx, ki67_col].X).flatten()
    
    # MKI67 and IDO1 transcripts from Xenium
    gene_data = {}
    for gene in XENIUM_GENES:
        if gene in rctd.var_names:
            expr = np.asarray(rctd[xen_valid_idx, gene].X.todense()).flatten() \
                if hasattr(rctd.X, 'todense') else np.asarray(rctd[xen_valid_idx, gene].X).flatten()
            gene_data[gene] = expr
        else:
            gene_data[gene] = np.full(n_valid, np.nan)
            print(f'  ⚠ {gene} not found in Xenium panel for {sample_name}')
    
    # RCTD cell type for each matched Xenium cell
    if RCTD_DOMINANT_COL in rctd.obs.columns:
        cell_types = rctd.obs.iloc[xen_valid_idx][RCTD_DOMINANT_COL].values
    else:
        cell_types = np.full(n_valid, 'unknown')
    
    # Ki-67 threshold (Otsu on arcsinh-transformed protein)
    all_ki67 = np.asarray(imc[:, ki67_col].layers['arcsinh']).flatten() \
        if 'arcsinh' in imc.layers else np.asarray(imc[:, ki67_col].X).flatten()
    try:
        ki67_threshold = threshold_otsu(all_ki67[all_ki67 > 0])
    except:
        ki67_threshold = np.median(all_ki67[all_ki67 > 0])
    ki67_pos = ki67_protein >= ki67_threshold
    
    # Store per-cell records
    for i in range(n_valid):
        rec = {
            'roi': roi_name,
            'sample': sample_name,
            'region': region,
            'distance': distances[valid_mask][i],
            'ki67_protein': ki67_protein[i],
            'ki67_pos': ki67_pos[i],
            'ki67_threshold': ki67_threshold,
            'cell_type': cell_types[i],
        }
        for gene in XENIUM_GENES:
            rec[f'{gene}_transcript'] = gene_data[gene][i]
        matched_records.append(rec)
    
    n_ki67_pos = ki67_pos.sum()
    print(f'  ✓ {roi_name} ({region}): {n_valid} matched, '
          f'{n_ki67_pos} Ki67+ ({100*n_ki67_pos/n_valid:.1f}%), '
          f'thresh={ki67_threshold:.2f}')

print(f'\nTotal: {len(matched_records)} matched cells, {n_skipped} ROIs skipped')

# Convert to DataFrame
df = pd.DataFrame(matched_records)
print(f'DataFrame shape: {df.shape}')
print(f'Regions: {df["region"].value_counts().to_dict()}')

## 3. MKI67 Transcript vs Ki-67 Protein Correlation

Test whether Xenium MKI67 transcript abundance correlates with IMC Ki-67
protein expression in spatially matched cells. This validates the IMC readout
used in Fig 6d and demonstrates cross-platform concordance.

In [ ]:
log_step('MKI67 transcript vs Ki-67 protein correlation')

# ── 3a. Per-ROI Spearman correlation ──
corr_results = []
for (roi, region), grp in df.groupby(['roi', 'region']):
    mki67_tx = grp['MKI67_transcript'].values
    ki67_pr = grp['ki67_protein'].values
    
    # Drop NaN
    valid = ~(np.isnan(mki67_tx) | np.isnan(ki67_pr))
    if valid.sum() < MIN_CELLS_PER_GROUP:
        continue
    
    rho, pval = stats.spearmanr(mki67_tx[valid], ki67_pr[valid])
    
    # Also: what fraction of Ki67+ cells have MKI67 > 0?
    ki67_pos_mask = grp['ki67_pos'].values[valid]
    mki67_in_pos = (mki67_tx[valid][ki67_pos_mask] > 0).mean() if ki67_pos_mask.sum() > 0 else np.nan
    mki67_in_neg = (mki67_tx[valid][~ki67_pos_mask] > 0).mean() if (~ki67_pos_mask).sum() > 0 else np.nan
    
    corr_results.append({
        'roi': roi, 'region': region, 'n_cells': valid.sum(),
        'spearman_rho': rho, 'pvalue': pval,
        'frac_mki67_in_ki67pos': mki67_in_pos,
        'frac_mki67_in_ki67neg': mki67_in_neg,
        'mean_protein_ki67pos': ki67_pr[valid][ki67_pos_mask].mean() if ki67_pos_mask.sum() > 0 else np.nan,
        'mean_transcript_ki67pos': mki67_tx[valid][ki67_pos_mask].mean() if ki67_pos_mask.sum() > 0 else np.nan,
    })

corr_df = pd.DataFrame(corr_results)

# BH correction
if len(corr_df) > 0:
    _, padj, _, _ = multipletests(corr_df['pvalue'].values, method='fdr_bh')
    corr_df['padj'] = padj

print(f'\nMKI67 transcript vs Ki-67 protein — per-ROI Spearman correlations:')
print(f'{"ROI":>30s}  {"Region":>6s}  {"n":>5s}  {"rho":>7s}  {"p":>10s}  {"padj":>10s}  {"MKI67+|Ki67+":>12s}  {"MKI67+|Ki67-":>12s}')
print('-' * 110)
for _, r in corr_df.iterrows():
    sig = '***' if r['padj'] < 0.001 else '**' if r['padj'] < 0.01 else '*' if r['padj'] < 0.05 else ''
    print(f'{r["roi"]:>30s}  {r["region"]:>6s}  {int(r["n_cells"]):>5d}  {r["spearman_rho"]:+.3f}  '
          f'{r["pvalue"]:.2e}  {r["padj"]:.2e}  {r["frac_mki67_in_ki67pos"]:>11.1%}  {r["frac_mki67_in_ki67neg"]:>11.1%}  {sig}')

# Summary
print(f'\nOverall: median rho = {corr_df["spearman_rho"].median():+.3f}')
print(f'  Core: median rho = {corr_df[corr_df["region"]=="Core"]["spearman_rho"].median():+.3f}')
print(f'  Edge: median rho = {corr_df[corr_df["region"]=="Edge"]["spearman_rho"].median():+.3f}')

corr_df.to_csv(os.path.join(outdir, '07_mki67_protein_transcript_corr.csv'), index=False)
print(f'\nSaved to {outdir}/07_mki67_protein_transcript_corr.csv')

In [ ]:
# ── 3b. Scatter plots: MKI67 transcript vs Ki-67 protein per ROI ──
n_rois = len(corr_df)
ncols = min(5, n_rois)
nrows = int(np.ceil(n_rois / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), squeeze=False)

for i, (_, row) in enumerate(corr_df.iterrows()):
    ax = axes[i // ncols, i % ncols]
    grp = df[df['roi'] == row['roi']]
    
    # Jitter MKI67 transcript counts slightly for visibility
    mki67 = grp['MKI67_transcript'].values + np.random.normal(0, 0.05, len(grp))
    ki67p = grp['ki67_protein'].values
    
    colors = ['#d62728' if pos else '#1f77b4' for pos in grp['ki67_pos']]
    ax.scatter(mki67, ki67p, c=colors, alpha=0.3, s=8, rasterized=True)
    
    ax.set_xlabel('MKI67 transcript', fontsize=8)
    ax.set_ylabel('Ki-67 protein (arcsinh)', fontsize=8)
    ax.set_title(f'{row["roi"]}\n{row["region"]} ρ={row["spearman_rho"]:+.3f}', fontsize=8)
    
    # Add threshold line
    thresh = grp['ki67_threshold'].iloc[0]
    ax.axhline(thresh, color='gray', linestyle='--', linewidth=0.5)

# Hide empty subplots
for j in range(i+1, nrows*ncols):
    axes[j // ncols, j % ncols].set_visible(False)

plt.suptitle('MKI67 Transcript vs Ki-67 Protein (red = Ki67+)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(outdir, '07_mki67_scatter_per_roi.pdf'))
plt.show()
print('Saved scatter plots')

## 4. IDO1 Expression in Ki-67+ vs Ki-67- Neighborhoods

**Hypothesis:** If IDO1 (tryptophan catabolism) is active in the tumor edge,
it depletes tryptophan and reduces de novo NAD+ synthesis, forcing proliferating
cells to rely on the Preiss-Handler pathway (nicotinic acid → NAD+). This would
explain the edge-specific enrichment of nicotinic acid near Ki-67+ cells.

We test:
1. Is IDO1 expression higher near Ki-67+ cells (same neighborhood analysis)?
2. Is IDO1 expression edge-specific?
3. Which cell types express IDO1?

In [ ]:
log_step('IDO1 analysis')

if 'IDO1_transcript' not in df.columns or df['IDO1_transcript'].isna().all():
    print('⚠ IDO1 not found in Xenium panel — skipping IDO1 analysis')
    print('  The Immuno-Oncology Profiling Panel (380 genes) DOES include IDO1.')
    print('  Check if the RCTD AnnData was built from the IO panel or the Brain panel.')
    print('  If IDO1 is not in the current AnnData, reload from the full Xenium output.')
else:
    # ── 4a. IDO1 expression: Ki67+ vs Ki67- cells ──
    ido1_results = []
    
    for region in ['Core', 'Edge']:
        reg = df[df['region'] == region]
        
        ki67_pos = reg[reg['ki67_pos']]
        ki67_neg = reg[~reg['ki67_pos']]
        
        # IDO1 expression in Ki67+ vs Ki67-
        ido1_pos = ki67_pos['IDO1_transcript'].dropna().values
        ido1_neg = ki67_neg['IDO1_transcript'].dropna().values
        
        if len(ido1_pos) >= 5 and len(ido1_neg) >= 5:
            # Mann-Whitney U test
            u_stat, u_pval = stats.mannwhitneyu(ido1_pos, ido1_neg, alternative='two-sided')
            
            ido1_results.append({
                'region': region,
                'n_ki67pos': len(ido1_pos),
                'n_ki67neg': len(ido1_neg),
                'mean_ido1_ki67pos': ido1_pos.mean(),
                'mean_ido1_ki67neg': ido1_neg.mean(),
                'frac_ido1_pos_in_ki67pos': (ido1_pos > 0).mean(),
                'frac_ido1_pos_in_ki67neg': (ido1_neg > 0).mean(),
                'mann_whitney_u': u_stat,
                'pvalue': u_pval,
            })
            
            print(f'\n{region}:')
            print(f'  Ki67+ cells: mean IDO1 = {ido1_pos.mean():.3f}, {100*(ido1_pos>0).mean():.1f}% expressing')
            print(f'  Ki67- cells: mean IDO1 = {ido1_neg.mean():.3f}, {100*(ido1_neg>0).mean():.1f}% expressing')
            print(f'  Mann-Whitney p = {u_pval:.2e}')
    
    ido1_ki67_df = pd.DataFrame(ido1_results)
    if len(ido1_ki67_df) > 0:
        _, padj, _, _ = multipletests(ido1_ki67_df['pvalue'].values, method='fdr_bh')
        ido1_ki67_df['padj'] = padj
    ido1_ki67_df.to_csv(os.path.join(outdir, '07_ido1_ki67_association.csv'), index=False)
    
    # ── 4b. IDO1 by cell type ──
    print(f'\nIDO1 expression by RCTD cell type (Edge):')
    edge_df = df[df['region'] == 'Edge']
    ct_ido1 = edge_df.groupby('cell_type')['IDO1_transcript'].agg(['mean', 'count', 
        lambda x: (x > 0).mean()]).rename(columns={'<lambda_0>': 'frac_expressing'})
    ct_ido1 = ct_ido1.sort_values('mean', ascending=False)
    print(ct_ido1.to_string())

## 5. IDO1 × Nicotinic Acid MSI Spatial Correlation

Match Xenium cells to nearest MSI pixels, then test whether IDO1 expression
correlates with local nicotinic acid MSI intensity.

In [ ]:
log_step('Matching Xenium cells to MSI pixels for IDO1 × nicotinic acid')

# Load MSI to find nicotinic acid channel(s)
first_msi = ad.read_h5ad(MSI_DIR / f'adata_{GBM_ROIS[first_roi]}.h5ad', backed='r')
print(f'MSI channels: {first_msi.n_vars}')
print(f'MSI obsm keys: {list(first_msi.obsm.keys())}')

# Determine MSI coordinate key — must be in H&E space
MSI_HE_CANDIDATES = ['spatial_he', 'spatial_warped_full_he']
MSI_COORD_KEY = None
for key in MSI_HE_CANDIDATES:
    if key in first_msi.obsm:
        MSI_COORD_KEY = key
        break
if MSI_COORD_KEY is None:
    raise KeyError(
        f'No H&E-space coordinates found in MSI AnnData. '
        f'Available obsm keys: {list(first_msi.obsm.keys())}.'
    )
print(f'Using MSI coords: obsm["{MSI_COORD_KEY}"]')

# Find nicotinic acid channels
nic_channels = []
for term in NICOTINIC_SEARCH_TERMS:
    matches = [v for v in first_msi.var_names if term.lower() in v.lower()]
    nic_channels.extend(matches)
nic_channels = list(set(nic_channels))
print(f'Nicotinic acid channels found: {nic_channels}')

if not nic_channels:
    print('⚠ No nicotinic acid channels found in MSI data')
    print('  Try checking channel names in MSI var_names manually:')
    named = [v for v in first_msi.var_names if not any(c.isdigit() for c in v[:3])]
    print(f'  Named channels: {named[:20]}...')
else:
    # Build Xenium → MSI matched dataframe with IDO1 and nicotinic acid
    nic_records = []
    
    for roi_name, sample_name in GBM_ROIS.items():
        region = 'Edge' if 'Edge' in sample_name else 'Core'
        
        msi_path = MSI_DIR / f'adata_{sample_name}.h5ad'
        rctd_path = RCTD_DIR / f'adata_{sample_name}_with_rctd.h5ad'
        
        if not msi_path.exists() or not rctd_path.exists():
            continue
        
        msi = ad.read_h5ad(msi_path)
        rctd = ad.read_h5ad(rctd_path)
        
        # Both must be in H&E space
        msi_coords = msi.obsm[MSI_COORD_KEY]
        xen_coords = rctd.obsm[XENIUM_COORD_KEY]
        
        # Match
        msi_tree = cKDTree(msi_coords)
        distances, msi_indices = msi_tree.query(xen_coords, k=1)
        valid = distances <= DISTANCE_CUTOFF_MSI
        
        if valid.sum() < MIN_CELLS_PER_GROUP:
            continue
        
        xen_idx = np.where(valid)[0]
        msi_idx = msi_indices[valid]
        
        # Get IDO1
        if 'IDO1' in rctd.var_names:
            ido1_vals = np.asarray(rctd[xen_idx, 'IDO1'].X.todense()).flatten() \
                if hasattr(rctd.X, 'todense') else np.asarray(rctd[xen_idx, 'IDO1'].X).flatten()
        else:
            ido1_vals = np.full(valid.sum(), np.nan)
        
        # Get nicotinic acid MSI
        for nic_ch in nic_channels:
            if nic_ch in msi.var_names:
                nic_vals = np.asarray(msi[msi_idx, nic_ch].X).flatten()
                
                for j in range(len(xen_idx)):
                    nic_records.append({
                        'roi': roi_name, 'sample': sample_name, 'region': region,
                        'nic_channel': nic_ch, 'nic_intensity': nic_vals[j],
                        'IDO1_transcript': ido1_vals[j],
                    })
        
        print(f'  ✓ {roi_name} ({region}): {valid.sum()} Xenium-MSI matches')
    
    nic_df = pd.DataFrame(nic_records)
    print(f'\nTotal Xenium-MSI matched: {len(nic_df)} cells')
    
    # ── 5b. Per-ROI Spearman: IDO1 vs nicotinic acid ──
    if len(nic_df) > 0 and not nic_df['IDO1_transcript'].isna().all():
        print(f'\nIDO1 × Nicotinic acid Spearman correlations:')
        ido1_nic_results = []
        
        for (roi, region, ch), grp in nic_df.groupby(['roi', 'region', 'nic_channel']):
            valid_mask = ~(grp['IDO1_transcript'].isna() | grp['nic_intensity'].isna())
            if valid_mask.sum() >= MIN_CELLS_PER_GROUP:
                rho, pval = stats.spearmanr(grp.loc[valid_mask, 'IDO1_transcript'],
                                            grp.loc[valid_mask, 'nic_intensity'])
                ido1_nic_results.append({
                    'roi': roi, 'region': region, 'nic_channel': ch,
                    'n': valid_mask.sum(), 'spearman_rho': rho, 'pvalue': pval
                })
        
        ido1_nic_df = pd.DataFrame(ido1_nic_results)
        if len(ido1_nic_df) > 0:
            _, padj, _, _ = multipletests(ido1_nic_df['pvalue'].values, method='fdr_bh')
            ido1_nic_df['padj'] = padj
            print(ido1_nic_df.to_string())
            ido1_nic_df.to_csv(os.path.join(outdir, '07_ido1_nicotinic_corr.csv'), index=False)
    else:
        print('⚠ Cannot compute IDO1 × nicotinic acid correlation (missing data)')

## 6. Summary Figure

In [ ]:
log_step('Generating summary figure')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Panel A: MKI67 transcript vs Ki-67 protein (all cells pooled) ──
ax = axes[0]
for region, color in [('Core', '#1f77b4'), ('Edge', '#d62728')]:
    subset = df[df['region'] == region]
    ax.scatter(subset['MKI67_transcript'] + np.random.normal(0, 0.05, len(subset)),
              subset['ki67_protein'], c=color, alpha=0.1, s=3, label=region, rasterized=True)

ax.set_xlabel('MKI67 transcript count')
ax.set_ylabel('Ki-67 protein (arcsinh)')
ax.set_title('A. MKI67 Transcript vs Ki-67 Protein')
ax.legend(frameon=False)

# ── Panel B: IDO1 expression in Ki67+ vs Ki67- by region ──
ax = axes[1]
if 'IDO1_transcript' in df.columns and not df['IDO1_transcript'].isna().all():
    plot_data = []
    for region in ['Core', 'Edge']:
        for ki67_status in [True, False]:
            subset = df[(df['region'] == region) & (df['ki67_pos'] == ki67_status)]
            vals = subset['IDO1_transcript'].dropna()
            plot_data.append({
                'Region': region,
                'Ki67': 'Ki67+' if ki67_status else 'Ki67-',
                'IDO1_mean': vals.mean(),
                'IDO1_sem': vals.sem(),
                'frac_expr': (vals > 0).mean(),
    })
    pd_plot = pd.DataFrame(plot_data)
    x = np.arange(len(pd_plot))
    colors = ['#d62728' if 'Ki67+' in r['Ki67'] else '#1f77b4' for _, r in pd_plot.iterrows()]
    ax.bar(x, pd_plot['IDO1_mean'], yerr=pd_plot['IDO1_sem'], color=colors, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['Region']}\n{r['Ki67']}" for _, r in pd_plot.iterrows()])
    ax.set_ylabel('Mean IDO1 transcript count')
    ax.set_title('B. IDO1 Expression by Ki-67 Status')
else:
    ax.text(0.5, 0.5, 'IDO1 not in Xenium panel', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('B. IDO1 Expression (unavailable)')

# ── Panel C: Per-ROI MKI67-Ki67 correlation by region ──
ax = axes[2]
if len(corr_df) > 0:
    for region, color in [('Core', '#1f77b4'), ('Edge', '#d62728')]:
        sub = corr_df[corr_df['region'] == region]
        ax.scatter(sub.index, sub['spearman_rho'], c=color, s=60, label=region, zorder=5)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_ylabel('Spearman ρ (MKI67 tx vs Ki67 protein)')
    ax.set_xlabel('ROI')
    ax.set_title('C. MKI67-Ki67 Correlation by ROI')
    ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(outdir, '07_summary_figure.pdf'))
plt.show()
print('Saved summary figure')

## 7. Key Takeaways

Print summary statistics for quick reference.

In [ ]:
log_step('Summary')

print('=' * 70)
print('P3-07 RESULTS SUMMARY')
print('=' * 70)

print(f'\n1. MKI67 TRANSCRIPT vs Ki-67 PROTEIN')
if len(corr_df) > 0:
    print(f'   {len(corr_df)} ROIs analyzed')
    print(f'   Median Spearman ρ = {corr_df["spearman_rho"].median():+.3f}')
    sig = corr_df[corr_df['padj'] < 0.05]
    print(f'   {len(sig)}/{len(corr_df)} ROIs significant (padj < 0.05)')
    print(f'   → {"VALIDATED" if len(sig) > len(corr_df)//2 else "PARTIAL"}: '
          f'IMC Ki-67 protein correlates with Xenium MKI67 transcript')

print(f'\n2. IDO1 ANALYSIS')
if 'IDO1_transcript' in df.columns and not df['IDO1_transcript'].isna().all():
    for region in ['Core', 'Edge']:
        reg = df[df['region'] == region]
        expr_rate = (reg['IDO1_transcript'] > 0).mean()
        print(f'   {region}: {100*expr_rate:.1f}% of cells express IDO1')
    
    # DC-specific
    dcs = df[df['cell_type'] == 'DC']
    if len(dcs) > 0:
        dc_edge = dcs[dcs['region'] == 'Edge']
        dc_core = dcs[dcs['region'] == 'Core']
        print(f'   DC cells at Edge: {100*(dc_edge["IDO1_transcript"]>0).mean():.1f}% express IDO1 (n={len(dc_edge)})')
        print(f'   DC cells at Core: {100*(dc_core["IDO1_transcript"]>0).mean():.1f}% express IDO1 (n={len(dc_core)})')
else:
    print('   IDO1 not available in Xenium panel')

print(f'\n3. IMPLICATIONS FOR FIGURE 6')
print(f'   If MKI67-Ki67 correlation is strong: validates IMC-based Ki-67 readout')
print(f'   If IDO1 is edge-enriched near Ki67+: supports NAD+ compensation hypothesis')
print(f'   Narrative: IDO1 → tryptophan depletion → reduced de novo NAD+')
print(f'             → compensatory nicotinic acid uptake (Preiss-Handler) near Ki67+ cells')
print('=' * 70)